# Variant prioritization - Filtering Variants after Annotation

This Jupyter Notebook uses Pandas to filter variants based on some rough clinical best practices
After the annotated data has been converted to a comma- or tab-separated files, common data wrangling languages like R or Python can be used to post-process dataframes

In [2]:
### First, let's load the dependencies

import pandas as pd ### Data Wrangling
import numpy as np ### Package to better handle numerical data


In [3]:
dataframe_annotated = pd.read_csv("/content/calls_annotated_true.txt",
                                    sep="\t",
                                  comment="#",
                                  on_bad_lines="skip") #deleted # in the 1st row of column names, bc "Only length-1 comment characters supported"
print(dataframe_annotated.head())

  Uploaded_variation     Location Allele             Gene          Feature  \
0    chr1_591525_A/G  chr1:591525      G  ENSG00000293331  ENST00000357876   
1    chr1_591525_A/G  chr1:591525      G  ENSG00000293331  ENST00000417636   
2    chr1_591525_A/G  chr1:591525      G  ENSG00000225880  ENST00000419394   
3    chr1_591525_A/G  chr1:591525      G  ENSG00000235146  ENST00000423796   
4    chr1_591525_A/G  chr1:591525      G  ENSG00000293331  ENST00000440196   

  Feature_type                                   Consequence cDNA_position  \
0   Transcript                       downstream_gene_variant             -   
1   Transcript                         upstream_gene_variant             -   
2   Transcript  intron_variant,non_coding_transcript_variant             -   
3   Transcript  intron_variant,non_coding_transcript_variant             -   
4   Transcript                       downstream_gene_variant             -   

  CDS_position Protein_position  ...  MAX_AF_POPS CLIN_SIG SOM

In [4]:
dataframe_annotated.columns

Index(['Uploaded_variation', 'Location', 'Allele', 'Gene', 'Feature',
       'Feature_type', 'Consequence', 'cDNA_position', 'CDS_position',
       'Protein_position', 'Amino_acids', 'Codons', 'Existing_variation',
       'IMPACT', 'DISTANCE', 'STRAND', 'FLAGS', 'VARIANT_CLASS', 'SYMBOL',
       'SYMBOL_SOURCE', 'HGNC_ID', 'BIOTYPE', 'CANONICAL', 'MANE',
       'MANE_SELECT', 'MANE_PLUS_CLINICAL', 'TSL', 'APPRIS', 'CCDS', 'ENSP',
       'SWISSPROT', 'TREMBL', 'UNIPARC', 'UNIPROT_ISOFORM', 'GENE_PHENO',
       'SIFT', 'PolyPhen', 'EXON', 'INTRON', 'DOMAINS', 'miRNA', 'HGVSc',
       'HGVSp', 'HGVS_OFFSET', 'AF', 'AFR_AF', 'AMR_AF', 'EAS_AF', 'EUR_AF',
       'SAS_AF', 'gnomADe_AF', 'gnomADe_AFR_AF', 'gnomADe_AMR_AF',
       'gnomADe_ASJ_AF', 'gnomADe_EAS_AF', 'gnomADe_FIN_AF', 'gnomADe_MID_AF',
       'gnomADe_NFE_AF', 'gnomADe_REMAINING_AF', 'gnomADe_SAS_AF',
       'gnomADg_AF', 'gnomADg_AFR_AF', 'gnomADg_AMI_AF', 'gnomADg_AMR_AF',
       'gnomADg_ASJ_AF', 'gnomADg_EAS_AF', 'gnomADg_F

In [5]:
### Let us also check for genes for which we do not have any HGNC genename
### In cases where no HGNC genename was assigned, we assume mutations to be deeply intronic or not associated to
### known functional structures of a gene
dataframe_annotated_firstpass = dataframe_annotated[dataframe_annotated["SYMBOL"] != '-']

### During annotation, it is expected that the majority of mutations do not contain database information
### These fields are either marked as "not accessible" (NA) or contain other placeholder string values

dataframe_annotated_secondpass = dataframe_annotated_firstpass[dataframe_annotated_firstpass["AF"] != '-']

dataframe_annotated_secondpass.head()

,Uploaded_variation,Location,Allele,Gene,Feature,Feature_type,Consequence,cDNA_position,CDS_position,Protein_position,...,MAX_AF_POPS,CLIN_SIG,SOMATIC,PHENO,PUBMED,MOTIF_NAME,MOTIF_POS,HIGH_INF_POS,MOTIF_SCORE_CHANGE,TRANSCRIPTION_FACTORS
36,chr1_629292_A/C,chr1:629292,C,ENSG00000237973,ENST00000414273,Transcript,upstream_gene_variant,-,-,-,...,gnomADg_AFR,-,-,-,-,-,-,-,-,-
37,chr1_629292_A/C,chr1:629292,C,ENSG00000225972,ENST00000416931,Transcript,non_coding_transcript_exon_variant,231,-,-,...,gnomADg_AFR,-,-,-,-,-,-,-,-,-
38,chr1_629292_A/C,chr1:629292,C,ENSG00000225880,ENST00000419394,Transcript,"intron_variant,non_coding_transcript_variant",-,-,-,...,gnomADg_AFR,-,-,-,-,-,-,-,-,-
39,chr1_629292_A/C,chr1:629292,C,ENSG00000229344,ENST00000427426,Transcript,upstream_gene_variant,-,-,-,...,gnomADg_AFR,-,-,-,-,-,-,-,-,-
41,chr1_629292_A/C,chr1:629292,C,ENSG00000225880,ENST00000440200,Transcript,"intron_variant,non_coding_transcript_variant",-,-,-,...,gnomADg_AFR,-,-,-,-,-,-,-,-,-


In [6]:
### In the next step, we will use the AF column to filter our data for potential germline variants
### The AF (Allele Frequency) is derived from the gnomAD population database and tells us how many (healthy) people
### harbor the corresponding mutation (in %)
### A mutation which is present at higher prevalence in the global population has a higher posterior chance of being
### non-pathogenic - e.g. if a TP53 mutation is present in 50 % of the global population, the chance of it being a
### major driver of oncogenecity is strongly decreased, as less than 50 % of the global population has cancer

### We first need to tell pandas that the column AF contains float values

dataframe_annotated_secondpass = dataframe_annotated_secondpass.astype({"AF": float})

dataframe_annotated_thirdpass = dataframe_annotated_secondpass[dataframe_annotated_secondpass["AF"] <= 0.02]
dataframe_annotated_thirdpass.head()


,Uploaded_variation,Location,Allele,Gene,Feature,Feature_type,Consequence,cDNA_position,CDS_position,Protein_position,...,MAX_AF_POPS,CLIN_SIG,SOMATIC,PHENO,PUBMED,MOTIF_NAME,MOTIF_POS,HIGH_INF_POS,MOTIF_SCORE_CHANGE,TRANSCRIPTION_FACTORS
36,chr1_629292_A/C,chr1:629292,C,ENSG00000237973,ENST00000414273,Transcript,upstream_gene_variant,-,-,-,...,gnomADg_AFR,-,-,-,-,-,-,-,-,-
37,chr1_629292_A/C,chr1:629292,C,ENSG00000225972,ENST00000416931,Transcript,non_coding_transcript_exon_variant,231,-,-,...,gnomADg_AFR,-,-,-,-,-,-,-,-,-
38,chr1_629292_A/C,chr1:629292,C,ENSG00000225880,ENST00000419394,Transcript,"intron_variant,non_coding_transcript_variant",-,-,-,...,gnomADg_AFR,-,-,-,-,-,-,-,-,-
39,chr1_629292_A/C,chr1:629292,C,ENSG00000229344,ENST00000427426,Transcript,upstream_gene_variant,-,-,-,...,gnomADg_AFR,-,-,-,-,-,-,-,-,-
41,chr1_629292_A/C,chr1:629292,C,ENSG00000225880,ENST00000440200,Transcript,"intron_variant,non_coding_transcript_variant",-,-,-,...,gnomADg_AFR,-,-,-,-,-,-,-,-,-


In [7]:
### After now filtering based on "missing information" and population allele frequency, we still have multiple entries
### per mutation
### This is due to the fact that each genomic position is associated to multiple transcripts, seen in the "Feature" column of
### the dataframe. The numbers shown here are the Ensembl Transcript identifiers, which are unique identifiers associated
### to each transcript of a gene (see Ensembl Gene Identifier under "Gene")

### The Ensembl consortium is only one of multiple consortia which are trying to unify nomenclature of genes and transcripts
### The NCBI database is also giving each gene/transcript another identifier, which can also be different from the HGVS consortium
### We are missing NCBI and HGVS annotations in our dataframe, as we were using the Ensembl VEP without additional parameters which
### activate annotation based on other consortia, but we have at least one additional helpful annotation source...

### The MANE consortium is trying to unify different annotations between all consortia and further tries to identify
### which transcript is the "primary transcript" for a gene (the transcript which is primarily produced in healthy cells after splicing)
### MANE also has an initiative in which it identifies specific transcripts as the "clinically most relevant" one by talking
### to clinicians and molecular pathologists

### Let's filter our data using the MANE Select and MANE PLUS CLINICAL identifiers

dataframe_annotated_clinical = dataframe_annotated_thirdpass[dataframe_annotated_thirdpass["MANE"].isin(["MANE_Select","MANE_Plus_Clinical"])]
dataframe_annotated_clinical

### ATTENTION: Based on experience, the MANE_Select transcript is NOT always the clinically significant one!
### A more sophisticated way of filtering would be to provide an NM-Transcript or ENST-Transcript list with the most relevant
### transcripts and filter based on that list



,Uploaded_variation,Location,Allele,Gene,Feature,Feature_type,Consequence,cDNA_position,CDS_position,Protein_position,...,MAX_AF_POPS,CLIN_SIG,SOMATIC,PHENO,PUBMED,MOTIF_NAME,MOTIF_POS,HIGH_INF_POS,MOTIF_SCORE_CHANGE,TRANSCRIPTION_FACTORS
2666,chr1_920105_G/A,chr1:920105,A,ENSG00000187634,ENST00000616016,Transcript,upstream_gene_variant,-,-,-,...,AFR,-,-,-,-,-,-,-,-,-
2859,chr1_928079_T/C,chr1:928079,C,ENSG00000187634,ENST00000616016,Transcript,intron_variant,-,-,-,...,EAS,-,-,-,-,-,-,-,-,-
2957,chr1_935493_G/A,chr1:935493,A,ENSG00000187634,ENST00000616016,Transcript,intron_variant,-,-,-,...,gnomADg_NFE,-,-,-,-,-,-,-,-,-
3041,chr1_938082_G/T,chr1:938082,T,ENSG00000187634,ENST00000616016,Transcript,intron_variant,-,-,-,...,gnomADg_AFR,-,-,-,-,-,-,-,-,-
3303,chr1_943364_G/C,chr1:943364,C,ENSG00000188976,ENST00000327044,Transcript,downstream_gene_variant,-,-,-,...,AMR,likely_benign,-,1,-,-,-,-,-,-
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2921085,chr12_85075397_C/T,chr12:85075397,T,ENSG00000133640,ENST00000393217,Transcript,intron_variant,-,-,-,...,AMR,-,-,-,-,-,-,-,-,-
2921089,chr12_85082916_A/G,chr12:85082916,G,ENSG00000133640,ENST00000393217,Transcript,intron_variant,-,-,-,...,gnomADg_AFR,-,-,-,-,-,-,-,-,-
2921091,chr12_85084463_T/A,chr12:85084463,A,ENSG00000133640,ENST00000393217,Transcript,intron_variant,-,-,-,...,AFR,-,-,-,-,-,-,-,-,-
2921107,chr12_85111304_T/A,chr12:85111304,A,ENSG00000133640,ENST00000393217,Transcript,intron_variant,-,-,-,...,AFR,-,-,-,-,-,-,-,-,-


In [ ]:
#51384 left

In [8]:
### Using the information from the clinical variation database ClinVar, a database of curated mutation pathogenicity
### associations, we can further drop mutations which have been clearly assigned as "benign"
### We keep mutations which are "likely benign" and do not filter based on a regex structure, as the ClinVar
### can assign a single mutation multiple clinical assessments. This is due to the fact that the same mutation can have different
### impacts on diverse morbidities, e.g. an oncogenic mutation alone can be assigned "likely_benign" in a hereditiary
### disease. We also keep unannotated mutations, as "missing" information in the ClinVar database does not imply
### a benign effect of the mutation

dataframe_annotated_clinical = dataframe_annotated_clinical[dataframe_annotated_clinical["CLIN_SIG"] != "benign"]
dataframe_annotated_clinical.head(3)


,Uploaded_variation,Location,Allele,Gene,Feature,Feature_type,Consequence,cDNA_position,CDS_position,Protein_position,...,MAX_AF_POPS,CLIN_SIG,SOMATIC,PHENO,PUBMED,MOTIF_NAME,MOTIF_POS,HIGH_INF_POS,MOTIF_SCORE_CHANGE,TRANSCRIPTION_FACTORS
2666,chr1_920105_G/A,chr1:920105,A,ENSG00000187634,ENST00000616016,Transcript,upstream_gene_variant,-,-,-,...,AFR,-,-,-,-,-,-,-,-,-
2859,chr1_928079_T/C,chr1:928079,C,ENSG00000187634,ENST00000616016,Transcript,intron_variant,-,-,-,...,EAS,-,-,-,-,-,-,-,-,-
2957,chr1_935493_G/A,chr1:935493,A,ENSG00000187634,ENST00000616016,Transcript,intron_variant,-,-,-,...,gnomADg_NFE,-,-,-,-,-,-,-,-,-


In [9]:
dataframe_annotated_clinical = dataframe_annotated_clinical[dataframe_annotated_clinical["SIFT"] != "tolerated(1)"]
dataframe_annotated_clinical

,Uploaded_variation,Location,Allele,Gene,Feature,Feature_type,Consequence,cDNA_position,CDS_position,Protein_position,...,MAX_AF_POPS,CLIN_SIG,SOMATIC,PHENO,PUBMED,MOTIF_NAME,MOTIF_POS,HIGH_INF_POS,MOTIF_SCORE_CHANGE,TRANSCRIPTION_FACTORS
2666,chr1_920105_G/A,chr1:920105,A,ENSG00000187634,ENST00000616016,Transcript,upstream_gene_variant,-,-,-,...,AFR,-,-,-,-,-,-,-,-,-
2859,chr1_928079_T/C,chr1:928079,C,ENSG00000187634,ENST00000616016,Transcript,intron_variant,-,-,-,...,EAS,-,-,-,-,-,-,-,-,-
2957,chr1_935493_G/A,chr1:935493,A,ENSG00000187634,ENST00000616016,Transcript,intron_variant,-,-,-,...,gnomADg_NFE,-,-,-,-,-,-,-,-,-
3041,chr1_938082_G/T,chr1:938082,T,ENSG00000187634,ENST00000616016,Transcript,intron_variant,-,-,-,...,gnomADg_AFR,-,-,-,-,-,-,-,-,-
3303,chr1_943364_G/C,chr1:943364,C,ENSG00000188976,ENST00000327044,Transcript,downstream_gene_variant,-,-,-,...,AMR,likely_benign,-,1,-,-,-,-,-,-
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2921085,chr12_85075397_C/T,chr12:85075397,T,ENSG00000133640,ENST00000393217,Transcript,intron_variant,-,-,-,...,AMR,-,-,-,-,-,-,-,-,-
2921089,chr12_85082916_A/G,chr12:85082916,G,ENSG00000133640,ENST00000393217,Transcript,intron_variant,-,-,-,...,gnomADg_AFR,-,-,-,-,-,-,-,-,-
2921091,chr12_85084463_T/A,chr12:85084463,A,ENSG00000133640,ENST00000393217,Transcript,intron_variant,-,-,-,...,AFR,-,-,-,-,-,-,-,-,-
2921107,chr12_85111304_T/A,chr12:85111304,A,ENSG00000133640,ENST00000393217,Transcript,intron_variant,-,-,-,...,AFR,-,-,-,-,-,-,-,-,-


In [10]:
dataframe_annotated_clinical[dataframe_annotated_clinical["SIFT"].str.contains("tolerated", na=False)]["SIFT"].value_counts().sort_index().head(100)

,count
SIFT,
tolerated(0.05),14
tolerated(0.06),27
tolerated(0.07),23
tolerated(0.08),26
tolerated(0.09),22
...,...
tolerated_low_confidence(0.2),3
tolerated_low_confidence(0.22),2
tolerated_low_confidence(0.23),1


In [11]:
### Let's first check if we still have genomic positions which are assigned to different consequences or genes
### Remember: Genes can overlap and be read from the 5' to 3' or in the reverse direction with different
### transcripts produced!
dataframe_annotated_clinical.Location.duplicated().any()


np.True_

In [12]:
### Let's filter these variants into a new dataframe to investigate them
duplicated_entries = dataframe_annotated_clinical[dataframe_annotated_clinical["Location"].duplicated(keep=False) == True]
print(duplicated_entries.head())

### We see that a majority of these alterations are linked to intronic variants or promoter regions, which can often
### shared between multiple genes

print(duplicated_entries.Consequence.unique())

     Uploaded_variation     Location Allele             Gene          Feature  \
3303    chr1_943364_G/C  chr1:943364      C  ENSG00000188976  ENST00000327044   
3314    chr1_943364_G/C  chr1:943364      C  ENSG00000187634  ENST00000616016   
3555    chr1_944730_G/A  chr1:944730      A  ENSG00000188976  ENST00000327044   
3566    chr1_944730_G/A  chr1:944730      A  ENSG00000187634  ENST00000616016   
3723    chr1_946132_G/T  chr1:946132      T  ENSG00000188976  ENST00000327044   

     Feature_type              Consequence cDNA_position CDS_position  \
3303   Transcript  downstream_gene_variant             -            -   
3314   Transcript         missense_variant          2674         2165   
3555   Transcript       synonymous_variant          2230         2214   
3566   Transcript  downstream_gene_variant             -            -   
3723   Transcript           intron_variant             -            -   

     Protein_position  ... MAX_AF_POPS       CLIN_SIG SOMATIC PHENO PUBMED

In [13]:
### At this point, depending on the tumor type or the potential treatments options, filter procedures can get very elaborated
### For the sake of simplicity of this workshop, we assume a solid tumor where we are trying to find a clear drug target
### based on an actionable protein change
### We can use the Consequence column of our dataframe to select for mutations which we assume to be relevant for this case with
### a clear impact on the protein sequence: These are missense mutations, truncating mutations (stop_gained), stop loss mutations
### and frameshifts

### First, we define our filter terms based on the Ensembl Consequence prediction classification:
### https://www.ensembl.org/info/genome/variation/prediction/predicted_data.html

consequence_filter = ["stop_gained", "frameshift_variant", "stop_lost", "missense_variant", "inframe_insertion", "inframe_deletion"]


### We have to consider that some of these consequences can be compound consequences, so we need str-matching to ensure
### we do not "overfilter" our data

dataframe_filtered_clinical = pd.DataFrame(columns=dataframe_annotated_clinical.columns) # Pre-allocate empty DF

for consequence in consequence_filter:
    found_match = dataframe_annotated_clinical[dataframe_annotated_clinical["Consequence"].str.contains(consequence, case=False, na=False)]
    dataframe_filtered_clinical = pd.concat([dataframe_filtered_clinical, found_match], ignore_index=True)

dataframe_filtered_clinical

/tmp/ipykernel_424/527019122.py:21: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  dataframe_filtered_clinical = pd.concat([dataframe_filtered_clinical, found_match], ignore_index=True)


,Uploaded_variation,Location,Allele,Gene,Feature,Feature_type,Consequence,cDNA_position,CDS_position,Protein_position,...,MAX_AF_POPS,CLIN_SIG,SOMATIC,PHENO,PUBMED,MOTIF_NAME,MOTIF_POS,HIGH_INF_POS,MOTIF_SCORE_CHANGE,TRANSCRIPTION_FACTORS
0,chr1_3497075_G/A,chr1:3497075,A,ENSG00000162591,ENST00000356575,Transcript,stop_gained,3766,3526,1176,...,EUR,-,-,-,-,-,-,-,-,-
1,chr1_55046549_C/G,chr1:55046549,G,ENSG00000169174,ENST00000302118,Transcript,stop_gained,716,426,142,...,AFR,"uncertain_significance,association,benign/like...",-,"1,1","21862702,25412415,16554528,26902539,15654334,2...",-,-,-,-,-
2,chr1_157767441_C/A,chr1:157767441,A,ENSG00000132704,ENST00000361516,Transcript,stop_gained,1011,952,318,...,AMR,-,-,-,-,-,-,-,-,-
3,chr1_158606897_C/A,chr1:158606897,A,ENSG00000198967,ENST00000641002,Transcript,stop_gained,706,459,153,...,AFR,-,-,-,30290772,-,-,-,-,-
4,chr1_169981984_C/A,chr1:169981984,A,ENSG00000075945,ENST00000361580,Transcript,stop_gained,2014,1786,596,...,AFR,-,"0,1,1","0,1,1",-,-,-,-,-,-
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1183,chr12_82896304_G/A,chr12:82896304,A,ENSG00000179104,ENST00000321196,Transcript,missense_variant,1822,1141,381,...,gnomADg_SAS,-,-,"0,1","29671961,27311106,29309402,38769084",-,-,-,-,-
1184,chr12_85027967_A/G,chr12:85027967,G,ENSG00000231738,ENST00000532498,Transcript,missense_variant,297,196,66,...,AFR,-,-,-,-,-,-,-,-,-
1185,chr12_85044745_A/C,chr12:85044745,C,ENSG00000133640,ENST00000393217,Transcript,missense_variant,353,272,91,...,AFR,-,-,-,-,-,-,-,-,-
1186,chr12_85056465_G/A,chr12:85056465,A,ENSG00000133640,ENST00000393217,Transcript,missense_variant,1753,1672,558,...,gnomADg_MID,-,"0,1,1","0,1,1",-,-,-,-,-,-


In [14]:
### We are getting to a point where clinicians are able to handle the amount of data in relation to the expected turn-
### around time for treatment!
### Since we are looking for actionable genes, we use the OncoKB database to further filter our data

### Load OncoKB cancer gene list
oncokb = pd.read_csv('/content/cancerGeneList.tsv', delimiter="\t+")
oncokb


/tmp/ipykernel_424/689080486.py:6: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  oncokb = pd.read_csv('/content/cancerGeneList.tsv', delimiter="\t+")


,Hugo Symbol,Entrez Gene ID,GRCh37 Isoform,GRCh37 RefSeq,GRCh38 Isoform,GRCh38 RefSeq,Gene Type,# of occurrence within resources (Column J-P),OncoKB Annotated,MSK-IMPACT,MSK-HEME,FOUNDATION ONE,FOUNDATION ONE HEME,Vogelstein,COSMIC CGC (v99),Gene Aliases
0,ABL1,25,ENST00000318560,NM_005157.4,ENST00000318560,NM_005157.4,ONCOGENE,7,Yes,Yes,Yes,Yes,Yes,Yes,Yes,"ABL, JTK7, c-ABL"
1,AKT1,207,ENST00000349310,NM_001014431.1,ENST00000349310,NM_001014431.1,ONCOGENE,7,Yes,Yes,Yes,Yes,Yes,Yes,Yes,"AKT, PKB, PRKBA, RAC, RAC-alpha"
2,ALK,238,ENST00000389048,NM_004304.4,ENST00000389048,NM_004304.4,ONCOGENE,7,Yes,Yes,Yes,Yes,Yes,Yes,Yes,CD246
3,AMER1,139285,ENST00000330258,NM_152424.3,ENST00000374869,NM_152424.3,TSG,7,Yes,Yes,Yes,Yes,Yes,Yes,Yes,"FAM123B, FLJ39827, RP11-403E24.2, WTX"
4,APC,324,ENST00000257430,NM_000038.5,ENST00000257430,NM_000038.5,TSG,7,Yes,Yes,Yes,Yes,Yes,Yes,Yes,"DP2.5, PPP1R46"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1216,ZFP36L2,678,ENST00000282388,NM_006887.4,ENST00000282388,NM_006887.4,TSG,1,Yes,No,No,No,No,No,No,"ERF2, RNF162C, TIS11D"
1217,ZNF24,7572,1,No,No,No,No,Yes,No,No,"KOX17, ZNF191, ZSCAN3, Zfp191",None,None,None,None,None
1218,ZNF292,23036,ENST00000369577,NM_015021.1,ENST00000369577,NM_015021.3,TSG,1,Yes,No,No,No,No,No,No,"KIAA0530, ZFP292, Zn-15, Zn-16, bA393I2.3"
1219,ZNF331,55422,1,No,No,No,No,No,No,Yes,"RITA, ZNF361, ZNF463",None,None,None,None,None


In [15]:
### We now filter our remaining calls for presence in the OncoKB database

dataframe_final_clinical = dataframe_filtered_clinical[dataframe_filtered_clinical["SYMBOL"].isin(oncokb["Hugo Symbol"])]


In [16]:
### And here we go! We filtered our (non-technically filtered) initial dataframe to a set of 14 mutations which
### are associated to cancer and oncogenesis. These mutations can further be processed based on the specific use case
### but would otherwise go to the clinicans for a literature search to identify potential targets!

dataframe_final_clinical

,Uploaded_variation,Location,Allele,Gene,Feature,Feature_type,Consequence,cDNA_position,CDS_position,Protein_position,...,MAX_AF_POPS,CLIN_SIG,SOMATIC,PHENO,PUBMED,MOTIF_NAME,MOTIF_POS,HIGH_INF_POS,MOTIF_SCORE_CHANGE,TRANSCRIPTION_FACTORS
32,chr1_1754585_C/T,chr1:1754585,T,ENSG00000008130,ENST00000341426,Transcript,missense_variant,1024,802,268,...,gnomADe_AFR,-,-,-,-,-,-,-,-,-
51,chr1_7663533_A/G,chr1:7663533,G,ENSG00000171735,ENST00000303635,Transcript,missense_variant,1063,986,329,...,gnomADe_EAS,"likely_benign,benign","0,1","1,1",-,-,-,-,-,-
52,chr1_7751246_C/A,chr1:7751246,A,ENSG00000171735,ENST00000303635,Transcript,missense_variant,4814,4737,1579,...,EAS,-,-,-,-,-,-,-,-,-
66,chr1_11790693_G/A,chr1:11790693,A,ENSG00000177000,ENST00000376590,Transcript,missense_variant,2048,1958,653,...,gnomADg_AMI,"likely_benign,benign","0,0,1","1,1,1","22241680,19421414,23227261",-,-,-,-,-
82,chr1_15928723_T/C,chr1:15928723,C,ENSG00000065526,ENST00000375759,Transcript,missense_variant,2844,2483,828,...,AMR,uncertain_significance,-,1,-,-,-,-,-,-
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1170,chr12_70590176_A/G,chr12:70590176,G,ENSG00000127329,ENST00000334414,Transcript,missense_variant,1872,1838,613,...,gnomADe_AFR,-,-,-,-,-,-,-,-,-
1171,chr12_70609137_T/C,chr12:70609137,C,ENSG00000127329,ENST00000334414,Transcript,missense_variant,945,911,304,...,SAS,-,-,-,-,-,-,-,-,-
1173,chr12_71580374_C/G,chr12:71580374,G,ENSG00000139292,ENST00000266674,Transcript,missense_variant,1786,1503,501,...,gnomADe_AMR,-,-,-,-,-,-,-,-,-
1174,chr12_71584265_A/G,chr12:71584265,G,ENSG00000139292,ENST00000266674,Transcript,missense_variant,2538,2255,752,...,EAS,-,"0,1","0,1",-,-,-,-,-,-


In [17]:
dataframe_final_clinical['IMPACT'].unique()

array(['MODERATE'], dtype=object)

In [18]:
dataframe_final_clinical["SIFT"].value_counts().sort_index()

,count
SIFT,
-,1
deleterious(0),5
deleterious(0.01),5
deleterious(0.02),3
deleterious(0.03),2
deleterious(0.04),2
deleterious_low_confidence(0),7
deleterious_low_confidence(0.01),5
deleterious_low_confidence(0.02),1


In [19]:
dataframe_final_clinical["PolyPhen"].value_counts().sort_index()

,count
PolyPhen,
-,9
benign(0),10
benign(0.001),1
benign(0.003),2
benign(0.005),1
benign(0.007),1
benign(0.009),1
benign(0.01),1
benign(0.014),1


In [20]:
dataframe_final_clinical["PolyPhen_score"] = dataframe_final_clinical["PolyPhen"].str.extract(r'\(([\d\.]+)\)').astype(float)
dataframe_final_clinical["PolyPhen_score"]

/tmp/ipykernel_424/413653291.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataframe_final_clinical["PolyPhen_score"] = dataframe_final_clinical["PolyPhen"].str.extract(r'\(([\d\.]+)\)').astype(float)


,PolyPhen_score
32,0.003
51,0.981
52,0.999
66,0.000
82,0.000
...,...
1170,0.000
1171,0.767
1173,0.219
1174,0.000
